In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_PATH = Path("..")
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
RAW_PATH = PROJECT_PATH / "data" / "raw"

print("Project path:", PROJECT_PATH.resolve())
print("Processed path:", PROCESSED_PATH.resolve())
print("Raw path:", RAW_PATH.resolve())

Project path: D:\fraud-loss-analysis-step1\fraud-loss-analysis
Processed path: D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\processed
Raw path: D:\fraud-loss-analysis-step1\fraud-loss-analysis\data\raw


In [3]:
# Load cleaned transactions
transactions = pd.read_csv(
    PROCESSED_PATH / "fact_transaction_clean.csv"
)

transactions["txn_ts"] = pd.to_datetime(
    transactions["txn_ts"],
    errors="coerce",
    format="mixed"
)

# Load chargebacks to reconstruct the confirmed fraud label
chargebacks = pd.read_csv(
    PROCESSED_PATH / "fact_chargeback.csv"
)

# Transactions linked to confirmed fraud chargebacks
fraud_txn_ids = set(
    chargebacks.loc[
        chargebacks["confirmed_fraud_flag"] == 1,
        "txn_id"
    ]
)

# Create fraud label for analysis
transactions["is_fraud"] = (
    transactions["txn_id"].isin(fraud_txn_ids)
).astype(int)

print("Transaction shape:", transactions.shape)

print("\nDate range:")
print("Start:", transactions["txn_ts"].min())
print("End:  ", transactions["txn_ts"].max())

print("\nFraud label distribution:")
print(transactions["is_fraud"].value_counts())

print("\nFraud rate:")
print(f"{transactions['is_fraud'].mean() * 100:.4f}%")

Transaction shape: (1498353, 15)

Date range:
Start: 2025-01-01 00:00:03
End:   2025-12-31 23:59:35

Fraud label distribution:
is_fraud
0    1487865
1      10488
Name: count, dtype: int64

Fraud rate:
0.7000%


In [4]:
# Basic transaction-time behavioral features

transactions["txn_date"] = transactions["txn_ts"].dt.date
transactions["txn_hour"] = transactions["txn_ts"].dt.hour

transactions["is_ecommerce"] = (
    transactions["channel"] == "E_COMMERCE"
).astype(int)

transactions["is_high_risk_category"] = (
    transactions["category"].isin(["Electronics", "Gift Cards"])
).astype(int)

transactions["is_country_mismatch"] = (
    (transactions["ip_country"] != transactions["merchant_country"]) &
    (transactions["ip_country"] != "UNKNOWN")
).astype(int)

transactions["is_small_amount"] = (
    transactions["amount"] <= 50
).astype(int)

transactions["is_high_value"] = (
    transactions["amount"] >= 20000
).astype(int)

# Sort chronologically within each card.
transactions = transactions.sort_values(
    ["card_id", "txn_ts"]
).reset_index(drop=True)

print("Feature preparation completed.")

print("\nNew behavioral features:")
print([
    "txn_date",
    "txn_hour",
    "is_ecommerce",
    "is_high_risk_category",
    "is_country_mismatch",
    "is_small_amount",
    "is_high_value"
])

print("\nTransaction shape:", transactions.shape)

Feature preparation completed.

New behavioral features:
['txn_date', 'txn_hour', 'is_ecommerce', 'is_high_risk_category', 'is_country_mismatch', 'is_small_amount', 'is_high_value']

Transaction shape: (1498353, 22)


In [5]:
# Create card-level transaction velocity features

transactions = transactions.sort_values(
    ["card_id", "txn_ts"]
).reset_index(drop=True)

# Use timestamp as the index for rolling calculations
card_groups = transactions.set_index("txn_ts").groupby("card_id")

transactions["txn_count_10m"] = (
    card_groups["txn_id"]
    .rolling("10min")
    .count()
    .reset_index(level=0, drop=True)
    .values
)

transactions["txn_count_1h"] = (
    card_groups["txn_id"]
    .rolling("1h")
    .count()
    .reset_index(level=0, drop=True)
    .values
)

transactions["txn_count_24h"] = (
    card_groups["txn_id"]
    .rolling("24h")
    .count()
    .reset_index(level=0, drop=True)
    .values
)

print("Velocity features created.")

print("\nVelocity statistics:")
print(
    transactions[
        ["txn_count_10m", "txn_count_1h", "txn_count_24h"]
    ].describe()
)

Velocity features created.

Velocity statistics:
       txn_count_10m  txn_count_1h  txn_count_24h
count   1,498,353.00  1,498,353.00   1,498,353.00
mean            1.00          1.01           1.14
std             0.03          0.08           0.37
min             1.00          1.00           1.00
25%             1.00          1.00           1.00
50%             1.00          1.00           1.00
75%             1.00          1.00           1.00
max             3.00          3.00           6.00


In [6]:
# Analyze small-amount transaction behavior

small_amount_summary = (
    transactions.groupby("is_small_amount")
    .agg(
        transactions=("txn_id", "count"),
        fraud_transactions=("is_fraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x: x[transactions.loc[x.index, "is_fraud"] == 1].sum())
    )
    .reset_index()
)

small_amount_summary["fraud_rate_pct"] = (
    small_amount_summary["fraud_transactions"]
    / small_amount_summary["transactions"]
    * 100
)

small_amount_summary["fraud_value_share_pct"] = (
    small_amount_summary["fraud_value"]
    / transactions.loc[transactions["is_fraud"] == 1, "amount"].sum()
    * 100
)

print("Small-amount transaction analysis:")
display(small_amount_summary)

Small-amount transaction analysis:


,is_small_amount,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct,fraud_value_share_pct
0,0,1495774,8458,"3,762,736,396.86","95,145,280.02",0.57,99.94
1,1,2579,2030,"74,227.13","52,580.32",78.71,0.06


In [7]:
# Focus on transactions where the card has another transaction
# within the previous 10 minutes.

small_burst = transactions[
    (transactions["is_small_amount"] == 1) &
    (transactions["txn_count_10m"] >= 2)
].copy()

print("\nSmall-amount + 10-minute velocity:")
print("Transactions:", len(small_burst))
print("Fraud transactions:", small_burst["is_fraud"].sum())

if len(small_burst) > 0:
    print(
        "Fraud rate:",
        f"{small_burst['is_fraud'].mean() * 100:.2f}%"
    )
    print(
        "Fraud value:",
        f"₹{small_burst.loc[small_burst['is_fraud'] == 1, 'amount'].sum():,.2f}"
    )
else:
    print("No small-amount bursts found.")


Small-amount + 10-minute velocity:
Transactions: 0
Fraud transactions: 0
No small-amount bursts found.


In [10]:
small_channel_analysis = (
    transactions[
        transactions["is_small_amount"] == 1
    ]
    .groupby("channel")
    .agg(
        transactions=("txn_id", "count"),
        fraud_transactions=("is_fraud", "sum"),
        transaction_value=("amount", "sum"),
        fraud_value=("amount", lambda x: x[
            transactions.loc[x.index, "is_fraud"] == 1
        ].sum())
    )
    .reset_index()
)

small_channel_analysis["fraud_rate_pct"] = (
    small_channel_analysis["fraud_transactions"]
    / small_channel_analysis["transactions"]
    * 100
)

small_channel_analysis["fraud_value_share_pct"] = (
    small_channel_analysis["fraud_value"]
    / transactions.loc[
        transactions["is_fraud"] == 1, "amount"
    ].sum()
    * 100
)

display(
    small_channel_analysis.sort_values(
        "fraud_rate_pct",
        ascending=False
    )
)
# Check the strongest small-amount combination:
# E-commerce + amount <= ₹50

small_ecommerce = transactions[
    (transactions["is_small_amount"] == 1) &
    (transactions["is_ecommerce"] == 1)
].copy()

print("\nSmall amount + E-commerce:")
print("Transactions:", len(small_ecommerce))
print("Fraud transactions:", small_ecommerce["is_fraud"].sum())

if len(small_ecommerce) > 0:
    print(
        "Fraud rate:",
        f"{small_ecommerce['is_fraud'].mean() * 100:.2f}%"
    )
    print(
        "Transaction value:",
        f"₹{small_ecommerce['amount'].sum():,.2f}"
    )
    print(
        "Fraud value:",
        f"₹{small_ecommerce.loc[small_ecommerce['is_fraud'] == 1, 'amount'].sum():,.2f}"
    )

,channel,transactions,fraud_transactions,transaction_value,fraud_value,fraud_rate_pct,fraud_value_share_pct
2,E_COMMERCE,2230,2030,"60,416.35","52,580.32",91.03,0.06
0,ATM,46,0,"1,822.66",0.00,0.00,0.00
1,CONTACTLESS,68,0,"2,662.68",0.00,0.00,0.00
3,POS,235,0,"9,325.44",0.00,0.00,0.00



Small amount + E-commerce:
Transactions: 2230
Fraud transactions: 2030
Fraud rate: 91.03%
Transaction value: ₹60,416.35
Fraud value: ₹52,580.32


In [11]:
# Previous transaction information for each card

transactions = transactions.sort_values(
    ["card_id", "txn_ts"]
).reset_index(drop=True)

transactions["previous_txn_ts"] = (
    transactions.groupby("card_id")["txn_ts"]
    .shift(1)
)

transactions["previous_merchant_country"] = (
    transactions.groupby("card_id")["merchant_country"]
    .shift(1)
)

transactions["previous_ip_country"] = (
    transactions.groupby("card_id")["ip_country"]
    .shift(1)
)

# Time between consecutive transactions for the same card
transactions["minutes_since_previous"] = (
    transactions["txn_ts"] - transactions["previous_txn_ts"]
).dt.total_seconds() / 60

# Detect a change in merchant country
transactions["country_changed"] = (
    transactions["merchant_country"]
    != transactions["previous_merchant_country"]
).astype(int)

print("Country-change features created.")

print("\nCountry changes:")
print(
    transactions["country_changed"]
    .value_counts()
    .sort_index()
)

print("\nTime-gap statistics:")
print(
    transactions["minutes_since_previous"]
    .describe()
)

Country-change features created.

Country changes:
country_changed
0    1157663
1     340690
Name: count, dtype: int64

Time-gap statistics:
count   1,468,353.00
mean       10,310.01
std        10,318.06
min             0.02
25%         2,968.63
50%         7,143.25
75%        14,284.57
max       181,054.28
Name: minutes_since_previous, dtype: float64


In [12]:
# Test different time thresholds for possible impossible travel

travel_thresholds = [30, 60, 120, 360, 1440]

travel_results = []

for threshold in travel_thresholds:
    
    mask = (
        (transactions["country_changed"] == 1) &
        (transactions["minutes_since_previous"] <= threshold)
    )
    
    subset = transactions[mask]
    
    fraud_transactions = subset["is_fraud"].sum()
    transaction_count = len(subset)
    fraud_value = subset.loc[
        subset["is_fraud"] == 1, "amount"
    ].sum()
    
    fraud_rate = (
        fraud_transactions / transaction_count * 100
        if transaction_count > 0 else 0
    )
    
    fraud_value_share = (
        fraud_value
        / transactions.loc[
            transactions["is_fraud"] == 1, "amount"
        ].sum()
        * 100
        if fraud_value > 0 else 0
    )
    
    travel_results.append({
        "threshold_minutes": threshold,
        "alert_volume": transaction_count,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": transaction_count - fraud_transactions,
        "fraud_rate_pct": fraud_rate,
        "fraud_value": fraud_value,
        "fraud_value_share_pct": fraud_value_share
    })

travel_analysis = pd.DataFrame(travel_results)

display(travel_analysis)

,threshold_minutes,alert_volume,fraud_transactions,legitimate_transactions,fraud_rate_pct,fraud_value,fraud_value_share_pct
0,30,962,10,952,1.04,"62,513.50",0.07
1,60,1781,28,1753,1.57,"122,071.01",0.13
2,120,3474,49,3425,1.41,"233,998.86",0.25
3,360,10445,108,10337,1.03,"519,831.46",0.55
4,1440,40275,415,39860,1.03,"2,576,614.99",2.71


In [13]:
# Analyze cross-border transactions combined with high-value behavior

cross_border_rules = {
    "country_mismatch": (
        transactions["is_country_mismatch"] == 1
    ),
    
    "country_mismatch_10k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["amount"] >= 10000)
    ),
    
    "country_mismatch_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["amount"] >= 20000)
    ),
    
    "ecommerce_country_mismatch": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1)
    ),
    
    "ecommerce_country_mismatch_10k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["amount"] >= 10000)
    ),
    
    "ecommerce_country_mismatch_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["amount"] >= 20000)
    )
}

cross_border_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for rule_name, mask in cross_border_rules.items():
    
    subset = transactions[mask]
    
    fraud_transactions = subset["is_fraud"].sum()
    alert_volume = len(subset)
    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()
    
    cross_border_results.append({
        "rule": rule_name,
        "alert_volume": alert_volume,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": alert_volume - fraud_transactions,
        "precision_pct": (
            fraud_transactions / alert_volume * 100
            if alert_volume > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

cross_border_analysis = pd.DataFrame(
    cross_border_results
)

display(cross_border_analysis)

,rule,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,country_mismatch,234667,2860,231807,1.22,"18,062,533.56",18.97
1,country_mismatch_10k,7146,437,6709,6.12,"12,559,054.23",13.19
2,country_mismatch_20k,1331,228,1103,17.13,"9,608,470.87",10.09
3,ecommerce_country_mismatch,90541,2548,87993,2.81,"15,714,062.76",16.51
4,ecommerce_country_mismatch_10k,2948,369,2579,12.52,"11,330,323.74",11.90
5,ecommerce_country_mismatch_20k,647,212,435,32.77,"9,102,756.03",9.56


In [14]:
# Cross-border + E-commerce + high-risk category analysis

cross_border_risk_rules = {
    "ecommerce_mismatch_risk_category": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["is_high_risk_category"] == 1)
    ),

    "ecommerce_mismatch_risk_category_10k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["is_high_risk_category"] == 1) &
        (transactions["amount"] >= 10000)
    ),

    "ecommerce_mismatch_risk_category_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["is_high_risk_category"] == 1) &
        (transactions["amount"] >= 20000)
    ),

    "ecommerce_mismatch_risk_category_25k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["is_high_risk_category"] == 1) &
        (transactions["amount"] >= 25000)
    )
}

cross_border_risk_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for rule_name, mask in cross_border_risk_rules.items():

    subset = transactions[mask]

    alert_volume = len(subset)
    fraud_transactions = subset["is_fraud"].sum()

    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()

    cross_border_risk_results.append({
        "rule": rule_name,
        "alert_volume": alert_volume,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": (
            alert_volume - fraud_transactions
        ),
        "precision_pct": (
            fraud_transactions / alert_volume * 100
            if alert_volume > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

cross_border_risk_analysis = pd.DataFrame(
    cross_border_risk_results
)

display(
    cross_border_risk_analysis
)

,rule,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,ecommerce_mismatch_risk_category,19080,749,18331,3.93,"11,347,786.79",11.92
1,ecommerce_mismatch_risk_category_10k,841,306,535,36.39,"10,361,950.20",10.88
2,ecommerce_mismatch_risk_category_20k,298,206,92,69.13,"8,899,134.50",9.35
3,ecommerce_mismatch_risk_category_25k,222,174,48,78.38,"8,183,107.98",8.60


In [15]:
# Development vs validation analysis for the complementary
# cross-border high-risk E-commerce rule

transactions["period"] = np.where(
    transactions["txn_ts"] < pd.Timestamp("2025-09-01"),
    "Development",
    "Validation"
)

thresholds = [10000, 15000, 20000, 25000, 30000]

dev_val_results = []

for threshold in thresholds:

    rule_mask = (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["is_high_risk_category"] == 1) &
        (transactions["amount"] >= threshold)
    )

    for period in ["Development", "Validation"]:

        subset = transactions[
            rule_mask &
            (transactions["period"] == period)
        ]

        alert_volume = len(subset)
        fraud_transactions = subset["is_fraud"].sum()

        fraud_value = subset.loc[
            subset["is_fraud"] == 1,
            "amount"
        ].sum()

        dev_val_results.append({
            "period": period,
            "threshold": threshold,
            "alert_volume": alert_volume,
            "fraud_transactions": fraud_transactions,
            "legitimate_transactions": (
                alert_volume - fraud_transactions
            ),
            "precision_pct": (
                fraud_transactions / alert_volume * 100
                if alert_volume > 0 else 0
            ),
            "fraud_value": fraud_value
        })

dev_val_analysis = pd.DataFrame(dev_val_results)

display(
    dev_val_analysis
    .sort_values(["threshold", "period"])
    .reset_index(drop=True)
)

,period,threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value
0,Development,10000,555,217,338,39.10,"7,018,688.53"
1,Validation,10000,286,89,197,31.12,"3,343,261.67"
2,Development,15000,303,172,131,56.77,"6,467,730.61"
3,Validation,15000,154,79,75,51.30,"3,217,659.55"
4,Development,20000,191,140,51,73.30,"5,899,881.33"
5,Validation,20000,107,66,41,61.68,"2,999,253.17"
6,Development,25000,147,118,29,80.27,"5,407,074.90"
7,Validation,25000,75,56,19,74.67,"2,776,033.08"
8,Development,30000,105,87,18,82.86,"4,562,534.84"
9,Validation,30000,54,44,10,81.48,"2,441,311.85"


In [16]:
# Business assumptions
REVIEW_COST_PER_ALERT = 150
CUSTOMER_FRICTION_PER_LEGIT_ALERT = 100
FRAUD_PREVENTION_RATE = 0.80
REVIEW_CAPACITY_PER_MONTH = 500

selected_candidates = []

for threshold in thresholds:

    validation_row = dev_val_analysis[
        (dev_val_analysis["period"] == "Validation") &
        (dev_val_analysis["threshold"] == threshold)
    ].iloc[0]

    alerts = validation_row["alert_volume"]
    fraud_transactions = validation_row["fraud_transactions"]
    legitimate_transactions = validation_row["legitimate_transactions"]
    fraud_value = validation_row["fraud_value"]
    precision = validation_row["precision_pct"]

    expected_fraud_prevented = (
        fraud_value * FRAUD_PREVENTION_RATE
    )

    review_cost = (
        alerts * REVIEW_COST_PER_ALERT
    )

    customer_friction_cost = (
        legitimate_transactions *
        CUSTOMER_FRICTION_PER_LEGIT_ALERT
    )

    expected_net_savings = (
        expected_fraud_prevented
        - review_cost
        - customer_friction_cost
    )

    monthly_alerts = alerts / 4

    capacity_utilization = (
        monthly_alerts /
        REVIEW_CAPACITY_PER_MONTH
        * 100
    )

    selected_candidates.append({
        "threshold": threshold,
        "validation_alerts": alerts,
        "validation_precision_pct": precision,
        "fraud_value_caught": fraud_value,
        "expected_fraud_prevented": expected_fraud_prevented,
        "review_cost": review_cost,
        "customer_friction_cost": customer_friction_cost,
        "expected_net_savings": expected_net_savings,
        "alerts_per_month": monthly_alerts,
        "capacity_utilization_pct": capacity_utilization
    })

complementary_economics = pd.DataFrame(
    selected_candidates
)

display(
    complementary_economics
    .sort_values("threshold")
    .reset_index(drop=True)
)

,threshold,validation_alerts,validation_precision_pct,fraud_value_caught,expected_fraud_prevented,review_cost,customer_friction_cost,expected_net_savings,alerts_per_month,capacity_utilization_pct
0,10000,286,31.12,"3,343,261.67","2,674,609.34",42900,19700,"2,612,009.34",71.50,14.30
1,15000,154,51.30,"3,217,659.55","2,574,127.64",23100,7500,"2,543,527.64",38.50,7.70
2,20000,107,61.68,"2,999,253.17","2,399,402.54",16050,4100,"2,379,252.54",26.75,5.35
3,25000,75,74.67,"2,776,033.08","2,220,826.46",11250,1900,"2,207,676.46",18.75,3.75
4,30000,54,81.48,"2,441,311.85","1,953,049.48",8100,1000,"1,943,949.48",13.50,2.70


In [17]:
# Selected complementary rule

COMPLEMENTARY_THRESHOLD = 25000

transactions["complementary_rule_alert"] = (
    (transactions["is_country_mismatch"] == 1) &
    (transactions["is_ecommerce"] == 1) &
    (transactions["is_high_risk_category"] == 1) &
    (transactions["amount"] >= COMPLEMENTARY_THRESHOLD)
).astype(int)

print("Selected complementary rule:")
print(
    "E_COMMERCE + country mismatch + "
    "Electronics/Gift Cards + amount >= ₹25,000"
)

print("\nRule alert count:")
print(
    transactions["complementary_rule_alert"].sum()
)

print("\nRule alert rate:")
print(
    f"{transactions['complementary_rule_alert'].mean() * 100:.4f}%"
)

Selected complementary rule:
E_COMMERCE + country mismatch + Electronics/Gift Cards + amount >= ₹25,000

Rule alert count:
222

Rule alert rate:
0.0148%


In [18]:
# Primary rule from Notebook 05
PRIMARY_THRESHOLD = 20000

transactions["primary_rule_alert"] = (
    (transactions["is_ecommerce"] == 1) &
    (transactions["is_high_risk_category"] == 1) &
    (transactions["amount"] >= PRIMARY_THRESHOLD)
).astype(int)

# Compare primary and complementary rules
rule_overlap = pd.crosstab(
    transactions["primary_rule_alert"],
    transactions["complementary_rule_alert"]
)

print("Rule overlap matrix:")
display(rule_overlap)

# Calculate overlap statistics
primary_alerts = transactions[
    transactions["primary_rule_alert"] == 1
]

complementary_alerts = transactions[
    transactions["complementary_rule_alert"] == 1
]

overlap_alerts = transactions[
    (transactions["primary_rule_alert"] == 1) &
    (transactions["complementary_rule_alert"] == 1)
]

complementary_only = transactions[
    (transactions["primary_rule_alert"] == 0) &
    (transactions["complementary_rule_alert"] == 1)
]

print("\nPrimary rule alerts:", len(primary_alerts))
print("Complementary rule alerts:", len(complementary_alerts))
print("Overlapping alerts:", len(overlap_alerts))
print("Complementary-only alerts:", len(complementary_only))

print(
    "\nComplementary rule overlap with primary:",
    f"{len(overlap_alerts) / len(complementary_alerts) * 100:.2f}%"
)

print(
    "Complementary rule adds new alerts:",
    f"{len(complementary_only) / len(complementary_alerts) * 100:.2f}%"
)

Rule overlap matrix:


complementary_rule_alert,0,1
primary_rule_alert,,
0,1496560,0
1,1571,222



Primary rule alerts: 1793
Complementary rule alerts: 222
Overlapping alerts: 222
Complementary-only alerts: 0

Complementary rule overlap with primary: 100.00%
Complementary rule adds new alerts: 0.00%


In [19]:
# The previous complementary rule is redundant with the primary rule.
# We keep it for diagnostic purposes but will NOT use it as a final rule.

transactions["merchant_card_count_24h"] = (
    transactions
    .groupby(["merchant_id", transactions["txn_ts"].dt.date])["card_id"]
    .transform("nunique")
)

# Number of unique cards using the same merchant on the same day
transactions["merchant_unique_cards_day"] = (
    transactions
    .groupby(
        ["merchant_id", transactions["txn_ts"].dt.date]
    )["card_id"]
    .transform("nunique")
)

print("Merchant/device behavioral features prepared.")

print("\nMerchant unique-card statistics:")
print(
    transactions["merchant_unique_cards_day"].describe()
)

Merchant/device behavioral features prepared.

Merchant unique-card statistics:
count   1,498,353.00
mean            3.05
std             1.43
min             1.00
25%             2.00
50%             3.00
75%             4.00
max            11.00
Name: merchant_unique_cards_day, dtype: float64


In [20]:
merchant_thresholds = [5, 6, 7, 8, 9, 10]

merchant_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for threshold in merchant_thresholds:

    mask = (
        transactions["merchant_unique_cards_day"] >= threshold
    )

    subset = transactions[mask]

    alert_volume = len(subset)
    fraud_transactions = subset["is_fraud"].sum()

    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()

    merchant_results.append({
        "unique_cards_threshold": threshold,
        "alert_volume": alert_volume,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": (
            alert_volume - fraud_transactions
        ),
        "precision_pct": (
            fraud_transactions / alert_volume * 100
            if alert_volume > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

merchant_concentration_analysis = pd.DataFrame(
    merchant_results
)

display(merchant_concentration_analysis)

,unique_cards_threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,5,227925,1618,226307,0.71,"13,717,527.54",14.41
1,6,85865,594,85271,0.69,"5,082,597.72",5.34
2,7,27139,199,26940,0.73,"1,616,698.99",1.70
3,8,7642,57,7585,0.75,"533,147.38",0.56
4,9,1850,11,1839,0.59,"89,186.85",0.09
5,10,374,1,373,0.27,"54,547.47",0.06


In [21]:
# Number of unique cards observed on each device within a calendar day

transactions["device_unique_cards_day"] = (
    transactions
    .groupby(
        ["device_id", transactions["txn_ts"].dt.date]
    )["card_id"]
    .transform("nunique")
)

print("Device unique-card statistics:")
print(
    transactions["device_unique_cards_day"].describe()
)

print("\nHigh device-card concentration candidates:")

device_thresholds = [3, 4, 5, 6, 7]

device_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for threshold in device_thresholds:

    mask = (
        transactions["device_unique_cards_day"] >= threshold
    )

    subset = transactions[mask]

    alert_volume = len(subset)
    fraud_transactions = subset["is_fraud"].sum()

    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()

    device_results.append({
        "unique_cards_threshold": threshold,
        "alert_volume": alert_volume,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": (
            alert_volume - fraud_transactions
        ),
        "precision_pct": (
            fraud_transactions / alert_volume * 100
            if alert_volume > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

device_concentration_analysis = pd.DataFrame(
    device_results
)

display(device_concentration_analysis)

Device unique-card statistics:
count   1,498,353.00
mean            1.13
std             0.51
min             1.00
25%             1.00
50%             1.00
75%             1.00
max            17.00
Name: device_unique_cards_day, dtype: float64

High device-card concentration candidates:


,unique_cards_threshold,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,3,12185,94,12091,0.77,"814,679.51",0.86
1,4,3290,31,3259,0.94,"185,838.31",0.20
2,5,2894,29,2865,1.00,"185,264.44",0.19
3,6,2724,27,2697,0.99,"180,332.07",0.19
4,7,2472,26,2446,1.05,"168,874.67",0.18


In [23]:
# Correct definition:
# A device is "new for card" only when this card has never
# used that device before the current transaction.

transactions = transactions.sort_values(
    ["card_id", "txn_ts"]
).reset_index(drop=True)

# Number of previous occurrences of this card-device combination
transactions["previous_card_device_count"] = (
    transactions
    .groupby(["card_id", "device_id"])
    .cumcount()
)

transactions["new_device_for_card"] = (
    transactions["previous_card_device_count"] == 0
).astype(int)

print("Corrected new-device feature.")

print("\nNew-device distribution:")
print(
    transactions["new_device_for_card"]
    .value_counts()
    .sort_index()
)

print("\nNew-device percentage:")
print(
    f"{transactions['new_device_for_card'].mean() * 100:.2f}%"
)

Corrected new-device feature.

New-device distribution:
new_device_for_card
0       1202
1    1497151
Name: count, dtype: int64

New-device percentage:
99.92%


In [24]:
# High-value E-commerce transactions outside
# Electronics and Gift Cards.

outside_primary_category = (
    transactions["category"].isin(
        ["Electronics", "Gift Cards"]
    ) == False
)

candidate_rules = {
    "ecommerce_high_value_other_category_10k": (
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 10000)
    ),

    "ecommerce_high_value_other_category_15k": (
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 15000)
    ),

    "ecommerce_high_value_other_category_20k": (
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 20000)
    ),

    "ecommerce_high_value_other_category_25k": (
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 25000)
    ),

    "ecommerce_high_value_other_category_30k": (
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 30000)
    )
}

incremental_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for rule_name, mask in candidate_rules.items():

    subset = transactions[mask]

    alert_volume = len(subset)
    fraud_transactions = subset["is_fraud"].sum()

    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()

    incremental_results.append({
        "rule": rule_name,
        "alert_volume": alert_volume,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": (
            alert_volume - fraud_transactions
        ),
        "precision_pct": (
            fraud_transactions / alert_volume * 100
            if alert_volume > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

incremental_category_analysis = pd.DataFrame(
    incremental_results
)

display(incremental_category_analysis)

,rule,alert_volume,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,ecommerce_high_value_other_category_10k,13691,237,13454,1.73,"3,913,443.22",4.11
1,ecommerce_high_value_other_category_15k,5098,99,4999,1.94,"2,265,104.56",2.38
2,ecommerce_high_value_other_category_20k,2306,43,2263,1.86,"1,294,064.13",1.36
3,ecommerce_high_value_other_category_25k,1176,21,1155,1.79,"796,436.01",0.84
4,ecommerce_high_value_other_category_30k,653,13,640,1.99,"584,385.21",0.61


In [25]:
# Candidate incremental rules
# All rules below are intentionally evaluated without hidden
# ground-truth typology information.

candidate_rules = {

    "cross_border_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["amount"] >= 20000)
    ),

    "ecommerce_cross_border_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        (transactions["amount"] >= 20000)
    ),

    "cross_border_non_primary_20k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["amount"] >= 20000) &
        outside_primary_category
    ),

    "cross_border_ecommerce_non_primary_10k": (
        (transactions["is_country_mismatch"] == 1) &
        (transactions["is_ecommerce"] == 1) &
        outside_primary_category &
        (transactions["amount"] >= 10000)
    ),

    "small_ecommerce": (
        (transactions["is_small_amount"] == 1) &
        (transactions["is_ecommerce"] == 1)
    )
}

candidate_results = []

total_fraud_value = transactions.loc[
    transactions["is_fraud"] == 1,
    "amount"
].sum()

for rule_name, mask in candidate_rules.items():

    subset = transactions[mask]

    alerts = len(subset)
    fraud_transactions = subset["is_fraud"].sum()

    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()

    candidate_results.append({
        "rule": rule_name,
        "alerts": alerts,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": alerts - fraud_transactions,
        "precision_pct": (
            fraud_transactions / alerts * 100
            if alerts > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value / total_fraud_value * 100
        )
    })

candidate_scorecard = pd.DataFrame(candidate_results)

display(
    candidate_scorecard.sort_values(
        "precision_pct",
        ascending=False
    ).reset_index(drop=True)
)

,rule,alerts,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,small_ecommerce,2230,2030,200,91.03,"52,580.32",0.06
1,ecommerce_cross_border_20k,647,212,435,32.77,"9,102,756.03",9.56
2,cross_border_20k,1331,228,1103,17.13,"9,608,470.87",10.09
3,cross_border_ecommerce_non_primary_10k,2107,63,2044,2.99,"968,373.54",1.02
4,cross_border_non_primary_20k,882,17,865,1.93,"566,526.66",0.60


In [26]:
# Final rule inventory for Notebook 07

rule_inventory = pd.DataFrame([
    {
        "rule_id": "R001",
        "rule_name": "High-value E-commerce high-risk category",
        "rule_type": "Primary loss-prevention rule",
        "condition": (
            "E_COMMERCE AND "
            "(Electronics OR Gift Cards) AND "
            "amount >= ₹20,000"
        ),
        "status": "RECOMMENDED",
        "reason": (
            "Strong validation economics and "
            "meets precision/capacity guardrails"
        )
    },
    {
        "rule_id": "R002",
        "rule_name": "E-commerce card-testing control",
        "rule_type": "Preventive control",
        "condition": (
            "E_COMMERCE AND amount <= ₹50"
        ),
        "status": "SECONDARY CONTROL",
        "reason": (
            "91.03% fraud rate but only 0.06% "
            "of total fraud value"
        )
    },
    {
        "rule_id": "R003",
        "rule_name": "Cross-border high-value",
        "rule_type": "Diagnostic/watchlist",
        "condition": (
            "Country mismatch AND amount >= ₹20,000"
        ),
        "status": "WATCHLIST",
        "reason": (
            "Meaningful fraud value but insufficient "
            "precision for standalone deployment"
        )
    },
    {
        "rule_id": "R004",
        "rule_name": "Impossible travel",
        "rule_type": "Diagnostic/watchlist",
        "condition": (
            "Merchant country changes within short "
            "time window"
        ),
        "status": "REJECTED",
        "reason": (
            "Low fraud-value coverage and weak "
            "operational economics"
        )
    },
    {
        "rule_id": "R005",
        "rule_name": "Merchant/device concentration",
        "rule_type": "Diagnostic/watchlist",
        "condition": (
            "Unusually high number of cards "
            "per merchant/device"
        ),
        "status": "REJECTED",
        "reason": (
            "Precision remains close to baseline"
        )
    }
])

display(rule_inventory)

,rule_id,rule_name,rule_type,condition,status,reason
0,R001,High-value E-commerce high-risk category,Primary loss-prevention rule,E_COMMERCE AND (Electronics OR Gift Cards) AND...,RECOMMENDED,Strong validation economics and meets precisio...
1,R002,E-commerce card-testing control,Preventive control,E_COMMERCE AND amount <= ₹50,SECONDARY CONTROL,91.03% fraud rate but only 0.06% of total frau...
2,R003,Cross-border high-value,Diagnostic/watchlist,"Country mismatch AND amount >= ₹20,000",WATCHLIST,Meaningful fraud value but insufficient precis...
3,R004,Impossible travel,Diagnostic/watchlist,Merchant country changes within short time window,REJECTED,Low fraud-value coverage and weak operational ...
4,R005,Merchant/device concentration,Diagnostic/watchlist,Unusually high number of cards per merchant/de...,REJECTED,Precision remains close to baseline


In [27]:
# ---------------------------------------------------------
# Combined operational impact of R001 + R002
# ---------------------------------------------------------

# R001: Primary loss-prevention rule
r001 = (
    (transactions["is_ecommerce"] == 1) &
    (transactions["is_high_risk_category"] == 1) &
    (transactions["amount"] >= 20000)
)

# R002: Card-testing preventive control
r002 = (
    (transactions["is_ecommerce"] == 1) &
    (transactions["amount"] <= 50)
)

transactions["R001_alert"] = r001.astype(int)
transactions["R002_alert"] = r002.astype(int)

# Analyze each rule independently
combined_results = []

for rule_id, rule_column in [
    ("R001", "R001_alert"),
    ("R002", "R002_alert")
]:
    
    subset = transactions[
        transactions[rule_column] == 1
    ]
    
    alerts = len(subset)
    fraud_transactions = subset["is_fraud"].sum()
    fraud_value = subset.loc[
        subset["is_fraud"] == 1,
        "amount"
    ].sum()
    
    combined_results.append({
        "rule_id": rule_id,
        "alerts": alerts,
        "fraud_transactions": fraud_transactions,
        "legitimate_transactions": (
            alerts - fraud_transactions
        ),
        "precision_pct": (
            fraud_transactions / alerts * 100
            if alerts > 0 else 0
        ),
        "fraud_value": fraud_value,
        "fraud_value_coverage_pct": (
            fraud_value /
            transactions.loc[
                transactions["is_fraud"] == 1,
                "amount"
            ].sum()
            * 100
        )
    })

combined_impact = pd.DataFrame(
    combined_results
)

display(combined_impact)

# ---------------------------------------------------------
# Check whether R001 and R002 overlap
# ---------------------------------------------------------

overlap = transactions[
    (transactions["R001_alert"] == 1) &
    (transactions["R002_alert"] == 1)
]

r001_only = transactions[
    (transactions["R001_alert"] == 1) &
    (transactions["R002_alert"] == 0)
]

r002_only = transactions[
    (transactions["R001_alert"] == 0) &
    (transactions["R002_alert"] == 1)
]

print("\nRule overlap:")
print("R001 + R002 overlap:", len(overlap))
print("R001 only:", len(r001_only))
print("R002 only:", len(r002_only))

print("\nR002 incremental fraud value:")
print(
    f"₹{r002_only.loc[r002_only['is_fraud'] == 1, 'amount'].sum():,.2f}"
)

print(
    "\nR002 incremental fraud transactions:",
    r002_only["is_fraud"].sum()
)

,rule_id,alerts,fraud_transactions,legitimate_transactions,precision_pct,fraud_value,fraud_value_coverage_pct
0,R001,1793,1318,475,73.51,"56,011,958.64",58.84
1,R002,2230,2030,200,91.03,"52,580.32",0.06



Rule overlap:
R001 + R002 overlap: 0
R001 only: 1793
R002 only: 2230

R002 incremental fraud value:
₹52,580.32

R002 incremental fraud transactions: 2030


In [28]:
# Save final rule inventory
rule_inventory_path = (
    PROCESSED_PATH / "rule_inventory.csv"
)

rule_inventory.to_csv(
    rule_inventory_path,
    index=False
)

# Save rule-level impact summary
combined_impact_path = (
    PROCESSED_PATH / "rule_impact_summary.csv"
)

combined_impact.to_csv(
    combined_impact_path,
    index=False
)

# Save candidate rule analysis
candidate_scorecard_path = (
    PROCESSED_PATH / "candidate_rule_scorecard.csv"
)

candidate_scorecard.to_csv(
    candidate_scorecard_path,
    index=False
)

# Save development/validation analysis
dev_val_path = (
    PROCESSED_PATH / "complementary_rule_dev_validation.csv"
)

dev_val_analysis.to_csv(
    dev_val_path,
    index=False
)

print("Saved outputs:")
print("1.", rule_inventory_path)
print("2.", combined_impact_path)
print("3.", candidate_scorecard_path)
print("4.", dev_val_path)

Saved outputs:
1. ..\data\processed\rule_inventory.csv
2. ..\data\processed\rule_impact_summary.csv
3. ..\data\processed\candidate_rule_scorecard.csv
4. ..\data\processed\complementary_rule_dev_validation.csv
